# Tiki Kitchen Appliances Market Analytics
## Data Cleaning

This notebook cleans the raw datasets based on the findings documented in
`data_check_report.md` (output of `data_quality_check.ipynb`).

Every transformation below is tied to a specific finding from the audit —
this notebook does not introduce new cleaning decisions that were not
already surfaced and reasoned about during the quality check.

### Steps
1. Setup
2. Load Data
3. Drop Unused Columns
4. Deduplication (defensive re-check)
5. Standardize Text Fields
6. Handle Missing Values by Business Meaning
7. Validate Price / Discount Logic (regression check)
8. Create Analytical Segments
9. Reconcile Review Counts
10. Save Cleaned Data
11. Cleaning Summary


### 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np
import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)


### 2. Load Data

In [2]:
RAW_PRODUCTS = "../data/raw/crawled_product_data.csv"
RAW_REVIEWS = "../data/raw/comments_data.csv"

OUT_PRODUCTS = "../data/processed/products_cleaned.csv"
OUT_REVIEWS = "../data/processed/reviews_cleaned.csv"

products = pd.read_csv(RAW_PRODUCTS)
reviews = pd.read_csv(RAW_REVIEWS)

n_products_before = len(products)
n_reviews_before = len(reviews)

print("Products:", products.shape)
print("Reviews:", reviews.shape)


Products: (2001, 30)
Reviews: (48649, 10)


In [3]:
mtime = os.path.getmtime("../data/raw/crawled_product_data.csv")
approx_crawl_date = datetime.datetime.fromtimestamp(mtime).date()
print(approx_crawl_date)

2026-08-15


In [4]:
products["snapshot_date"] = approx_crawl_date

### 3. Drop Unused Columns

`meta_title` was found to be 100% missing (0/2,001 non-null) during the
data quality audit. It carries no analytical value and is dropped here.

In [5]:
cols_to_drop = ["meta_title"]

products = products.drop(
    columns=[c for c in cols_to_drop if c in products.columns]
)

print("Dropped columns:", cols_to_drop)
print("Remaining columns:", products.shape[1])


Dropped columns: ['meta_title']
Remaining columns: 30


### 4. Deduplication (defensive re-check)

The audit reported 0 duplicate `id` / `comment_id` values. This step is
kept anyway as a defensive check — cleaning code should not blindly trust
a prior audit run against a possibly different raw file snapshot.

In [6]:
before_p = len(products)
products = products.drop_duplicates(subset=["id"])
print(f"Products deduped: {before_p} -> {len(products)} "
      f"(removed {before_p - len(products)})")

before_r = len(reviews)
reviews = reviews.drop_duplicates(subset=["comment_id"])
print(f"Reviews deduped: {before_r} -> {len(reviews)} "
      f"(removed {before_r - len(reviews)})")


Products deduped: 2001 -> 2001 (removed 0)
Reviews deduped: 48649 -> 48649 (removed 0)


### 5. Standardize Text Fields

Trim whitespace and normalize `NaN` representation for text fields that
will later be used as groupby keys (`brand_name`, `seller_name`,
`category_name`). Inconsistent whitespace/casing in these fields would
silently fragment groupby results in the KPI stage.

In [7]:
text_cols = ["brand_name", "seller_name", "category_name"]

for col in text_cols:
    if col in products.columns:
        products[col] = products[col].astype(str).str.strip()
        products[col] = products[col].replace({"nan": np.nan})

products[text_cols].head()


,brand_name,seller_name,category_name
0,Bluestone,Tiki Trading,Điện Gia Dụng
1,Sharp,Tiki Trading,CHƯƠNG TRÌNH KHUYẾN MẠI KHÁC
2,Sharp,Tiki Trading,Điện Gia Dụng
3,Cuckoo,Tiki Trading,Điện Gia Dụng
4,Philips,Tiki Trading,Máy ép nhanh


### 6. Handle Missing Values by Business Meaning

Two missing-value patterns were identified in the audit, each with a
different correct treatment:

- **`quantity_sold_value`** missing (~42.5% of rows) does not mean "sold
  0 units" — it means no sales have ever been recorded for that product.
  Blindly filling with 0 would erase this distinction. A `has_sold` flag
  is created instead, and a filled column is kept separately for
  aggregation convenience.
- **`seller_id` / `category_id`** missing (8 records) likely reflects
  products that became unavailable/delisted between crawl steps. Per the
  audit's Action Plan, these are flagged for manual review rather than
  dropped outright.

**Note added after further review:** `all_time_quantity_sold` was found to have 4 more non-null rows than `quantity_sold_value` (1154 vs 1150 in the audited raw data) — the two fields are *not* fully identical, despite looking the same at a glance. Rather than discarding `all_time_quantity_sold` outright (which would silently drop those 4 data points), it is used as a fallback to fill gaps in `quantity_sold_value` before being dropped as redundant.

In [8]:
# Coalesce quantity_sold_value with all_time_quantity_sold BEFORE
# computing has_sold — using quantity_sold_value alone would silently
# under-count has_sold for the ~4 rows where all_time_quantity_sold has
# a value but quantity_sold_value does not.

both_notna = (
    products["quantity_sold_value"].notna() & products["all_time_quantity_sold"].notna()
)
disagree_when_both_present = (
    products.loc[both_notna, "quantity_sold_value"]
    != products.loc[both_notna, "all_time_quantity_sold"]
).sum()
only_all_time_has_value = (
    products["all_time_quantity_sold"].notna() & products["quantity_sold_value"].isna()
).sum()

print(f"Rows where both fields present but disagree: {disagree_when_both_present}")
print(f"Rows rescued from all_time_quantity_sold (quantity_sold_value was missing): "
      f"{only_all_time_has_value}")

products["quantity_sold_value"] = products["quantity_sold_value"].fillna(
    products["all_time_quantity_sold"]
)

# Now fully absorbed into quantity_sold_value -> safe to drop
products = products.drop(columns=["all_time_quantity_sold"])

products["has_sold"] = products["quantity_sold_value"].notna()
products["quantity_sold_value_filled"] = products["quantity_sold_value"].fillna(0)

print("has_sold = True :", products["has_sold"].sum())
print("has_sold = False:", (~products["has_sold"]).sum())


Rows where both fields present but disagree: 0
Rows rescued from all_time_quantity_sold (quantity_sold_value was missing): 4
has_sold = True : 1154
has_sold = False: 847


**Note on `quantity_sold_text`:** this is the raw display string
(e.g. `"Đã bán 131"`) that `quantity_sold_value` was parsed from. It
carries no analytical value beyond what `quantity_sold_value_filled`
already provides, but it is kept (not dropped) for traceability / display
purposes (e.g. dashboard tooltips). Missing values here follow the exact
same pattern as `quantity_sold_value` — both are `NaN` together for
products with no recorded sales — so it is filled with an explicit
placeholder string rather than left blank, to avoid ambiguity between
"missing because unsold" and "missing because of a parsing error".

In [9]:
# Sanity check: quantity_sold_text and quantity_sold_value should be
# missing on exactly the same rows (both parsed from the same source field).
mismatch_na = (
    products["quantity_sold_text"].isna() != products["quantity_sold_value"].isna()
)
print(f"Rows where quantity_sold_text/value missingness disagrees: {mismatch_na.sum()}")

products["quantity_sold_text"] = products["quantity_sold_text"].fillna("Chua co luot ban")

print("quantity_sold_text missing after fill:", products["quantity_sold_text"].isna().sum())


Rows where quantity_sold_text/value missingness disagrees: 4
quantity_sold_text missing after fill: 0


In [10]:
products["possibly_delisted"] = (
    products["seller_id"].isna() | products["category_id"].isna()
)

print("possibly_delisted flagged:", products["possibly_delisted"].sum())
products.loc[products["possibly_delisted"], ["id", "product_name", "seller_id", "category_id"]]


possibly_delisted flagged: 8


,id,product_name,seller_id,category_id
234,49601294,Nồi cơm điện Tiger JNP-1000 (Màu trắng) - Hàng...,NaN,NaN
418,127031344,Máy Nướng Bánh Mì Kẹp BlueStone SBB-2333 (650W...,NaN,NaN
479,147903536,Bếp Từ Đơn Sunhouse SHD6803 (2000W) - Kèm Nồ...,NaN,NaN
728,205818604,[HÀNG CHÍNH HÃNG] Máy Ép Chậm Hurom H100S,NaN,NaN
1521,276849824,NỒI CHIÊN KHÔNG DẦU 9L SUNHOUSE SHD4037 - Hàng...,NaN,NaN
1600,277512215,Nồi cơm điện tử Kangaroo 1.5 lít KG15RCE2 - Hà...,NaN,NaN
1678,278096977,Máy xay sinh tố Locknlock 950ml Duo Turbo Blen...,NaN,NaN
1888,278922364,Máy pha viên nén Trà và Cafe Wells Home Cafe [...,NaN,NaN


### 7. Validate Price / Discount Logic (regression check)

`original_price` is used as the reference price (more reliable than
`list_price` in general, though the audit confirmed `list_price` is also
100% valid in this particular dataset). `discount_rate` is recomputed
independently and compared to the crawled value to catch inconsistencies.

The two `assert` checks re-verify that the validity findings from the
audit (0 invalid prices, 0 `original_price < price` logic errors) still
hold after the transformations above — cleaning code should not silently
break what was already confirmed clean.

**Bug fix (found during review):** the mismatch threshold was originally set to `> 5` percentage points, but `discount_rate` from the API is rounded to a whole number while `discount_rate_calc` is a continuous float — so the expected rounding error is at most ~0.5pp. A 5pp threshold is 10x looser than that, so the check could never flag anything regardless of whether real data errors existed. Tightened to 1.0pp (a small buffer above the theoretical 0.5pp max) so the check is actually meaningful.

In [11]:
products["reference_price"] = products["original_price"].where(
    products["original_price"] > 0, products["price"]
)

products["discount_rate_calc"] = (
    (products["reference_price"] - products["price"]) / products["reference_price"] * 100
)

products["discount_rate_diff"] = (
    products["discount_rate"] - products["discount_rate_calc"]
).abs()

print(products["discount_rate_diff"].describe())


count   2001.00
mean       0.08
std        0.37
min        0.00
25%        0.00
50%        0.00
75%        0.04
max       15.29
Name: discount_rate_diff, dtype: float64


In [12]:
# The API's discount_rate is rounded to a whole number, so pure rounding
# noise should never exceed ~0.5pp. Threshold set slightly above that to
# leave a small buffer while still being tight enough to catch real
# mismatches. (Previously set to >5pp, which was 10x looser than the max
# possible rounding error -> the check could never flag anything.)
MISMATCH_THRESHOLD = 1.0

products["discount_rate_mismatch"] = products["discount_rate_diff"] > MISMATCH_THRESHOLD

n_mismatch = products["discount_rate_mismatch"].sum()
print(f"discount_rate mismatch (>{MISMATCH_THRESHOLD}pp vs recalculated): {n_mismatch} "
      f"({n_mismatch / len(products) * 100:.1f}%)")


discount_rate mismatch (>1.0pp vs recalculated): 2 (0.1%)


In [13]:
assert (products["price"] <= 0).sum() == 0, "Found price <= 0 after cleaning!"
assert (products["original_price"] < products["price"]).sum() == 0, (
    "Found original_price < price after cleaning!"
)

print("Regression check passed: price / original_price logic still valid.")


Regression check passed: price / original_price logic still valid.


### 8. Create Analytical Segments

`price_bucket` and `discount_bucket` are created here so that EDA and
KPI notebooks downstream can reuse a single, consistent segmentation
instead of redefining bins in multiple places.

In [14]:
products["price_bucket"] = pd.cut(
    products["price"],
    bins=[0, 100_000, 300_000, 700_000, 1_500_000, float("inf")],
    labels=["<100k", "100-300k", "300-700k", "700k-1.5tr", ">1.5tr"],
)

products["discount_bucket"] = pd.cut(
    products["discount_rate"],
    bins=[-0.1, 0, 10, 20, 30, 50, 100],
    labels=["0%", "0-10%", "10-20%", "20-30%", "30-50%", ">50%"],
)

products[["price", "price_bucket", "discount_rate", "discount_bucket"]].head()


,price,price_bucket,discount_rate,discount_bucket
0,551000,300-700k,34,30-50%
1,756000,700k-1.5tr,20,10-20%
2,2061000,>1.5tr,21,20-30%
3,1490000,700k-1.5tr,35,30-50%
4,932000,700k-1.5tr,36,30-50%


**`subcategory_proxy`:** moved here from the EDA notebook, where it was
originally computed in-memory only and never persisted — meaning it
never actually reached `products_cleaned.csv`, the warehouse ETL, or the
dashboard. Since this is a feature-engineering step (same category as
`price_bucket`/`discount_bucket` above), it belongs in cleaning, not in
an exploratory notebook, so downstream steps (EDA, KPI, warehouse
loading) can all read it as a normal column instead of recomputing it
in multiple places.

**Known limitation (unchanged from EDA):** this is a keyword-based
heuristic tagged from `product_name`, not Tiki's official taxonomy.
Order matters — specific appliance types are matched before the generic
catch-all bucket, otherwise e.g. "nồi chiên không dầu" would be
swallowed by the generic "nồi " keyword.

In [15]:
# Interim proxy: rough sub-category tagging from product_name keywords.
# NOTE: this is a simple heuristic, not a validated taxonomy.
#
# IMPORTANT: order matters. More SPECIFIC categories must be checked
# BEFORE generic ones — e.g. "nồi chiên không dầu" / "nồi cơm điện"
# must be matched before the generic "Nồi/Chảo" bucket, otherwise the
# generic "nồi " keyword would swallow them first.

keyword_map = [
    # --- Specific appliance types (check first) ---
    ("Nồi chiên không dầu", ["nồi chiên không dầu", "air fryer"]),
    ("Nồi cơm điện", ["nồi cơm điện", "nồi cơm"]),
    ("Nồi áp suất", ["nồi áp suất"]),
    ("Nồi lẩu điện", ["nồi lẩu"]),
    ("Máy pha cà phê", ["máy pha cà phê", "máy pha cafe", "máy pha coffee"]),
    ("Máy làm sữa hạt", ["máy làm sữa hạt"]),
    ("Máy làm sữa chua", ["máy làm sữa chua", "máy ủ sữa chua"]),
    ("Máy đánh trứng", ["máy đánh trứng"]),
    ("Máy xay", ["máy xay", "xay sinh tố", "xay cầm tay", "xay đa năng"]),
    ("Máy ép", ["máy ép trái cây", "máy ép chậm", "máy ép"]),
    ("Bếp điện/từ", ["bếp điện", "bếp từ", "bếp hồng ngoại", "bếp gas", "bếp nướng"]),
    ("Lò vi sóng/nướng", ["lò vi sóng", "lò nướng"]),
    ("Ấm đun nước/Bình siêu tốc", ["ấm đun", "ấm siêu tốc", "ấm điện", "bình đun siêu tốc"]),
    ("Bình giữ nhiệt", ["bình giữ nhiệt", "bình thủy", "ly giữ nhiệt", "ca giữ nhiệt"]),
    ("Máy lọc nước/Cây nước nóng lạnh", ["máy lọc nước", "cây nước nóng lạnh", "bình lọc nước"]),
    ("Máy rửa chén/bát", ["máy rửa chén", "máy rửa bát"]),
    ("Máy hút mùi", ["máy hút mùi", "hút mùi"]),
    ("Máy hút chân không", ["máy hút chân không", "hút chân không"]),
    ("Máy làm bánh mì", ["máy làm bánh mì"]),
    ("Máy xay thịt/Máy xay đa năng", ["máy xay thịt"]),
    ("Tủ lạnh/Tủ mát mini", ["tủ lạnh mini", "tủ mát mini", "tủ đông mini"]),
    ("Dao/Kéo/Thớt bếp", ["dao bếp", "bộ dao", "kéo bếp", "thớt "]),
    ("Hộp đựng thực phẩm", ["hộp đựng thực phẩm", "hộp cơm", "hộp bảo quản"]),
    ("Khuôn/Dụng cụ làm bánh", ["khuôn bánh", "khuôn làm bánh", "dụng cụ làm bánh"]),

    # --- Generic catch-alls (check last) ---
    ("Nồi/Chảo (khác)", ["nồi ", "chảo ", "xoong "]),
]

def tag_subcategory(name):
    name_lower = f" {str(name).lower()} "  # padding để tránh match nhầm ký tự dính từ
    for label, keywords in keyword_map:
        if any(kw in name_lower for kw in keywords):
            return label
    return "Khác / Chưa phân loại"

products["subcategory_proxy"] = products["product_name"].apply(tag_subcategory)

subcat_counts = products["subcategory_proxy"].value_counts()
unclassified_pct = (
    (products["subcategory_proxy"] == "Khác / Chưa phân loại").mean() * 100
)
print(f"Unclassified rate: {unclassified_pct:.1f}%")
subcat_counts


Unclassified rate: 15.4%


subcategory_proxy
Khác / Chưa phân loại        309
Nồi cơm điện                 274
Máy xay                      262
Bếp điện/từ                  244
Ấm đun nước/Bình siêu tốc    162
Nồi/Chảo (khác)              129
Máy pha cà phê                93
Nồi chiên không dầu           84
Máy ép                        80
Lò vi sóng/nướng              78
Máy làm sữa hạt               68
Nồi lẩu điện                  66
Nồi áp suất                   44
Máy đánh trứng                29
Bình giữ nhiệt                20
Máy hút mùi                   19
Máy hút chân không            16
Máy làm sữa chua              13
Hộp đựng thực phẩm             7
Máy làm bánh mì                3
Máy rửa chén/bát               1
Name: count, dtype: int64

### 9. Reconcile Review Counts

The audit found that `review_count` (a static field on the product page)
can diverge substantially from the actual number of reviews collected by
the crawler — most noticeably for product `id 392842` (declared: 3,
crawled: 170). The crawled count is treated as ground truth, since it
reflects data measured directly rather than a potentially stale page
field.

`crawled_review_count` is computed here and merged into `products`;
`review_count` is kept as-is (not overwritten) so both can be compared
later if needed.

In [16]:
crawled_review_counts = (
    reviews.groupby("product_id").size().reset_index(name="crawled_review_count")
)

products = products.merge(
    crawled_review_counts, left_on="id", right_on="product_id", how="left"
)
products["crawled_review_count"] = products["crawled_review_count"].fillna(0).astype(int)
products = products.drop(columns=["product_id"])

products[["id", "review_count", "crawled_review_count"]].head()


,id,review_count,crawled_review_count
0,359479,1232,1245
1,368305,40,48
2,368309,47,59
3,374109,10,17
4,392729,371,400


In [17]:
large_mismatch = products[
    (products["review_count"] > 0)
    & (products["crawled_review_count"] > 0)
    & (
        (products["crawled_review_count"] / products["review_count"].replace(0, np.nan))
        > 3
    )
]

print(f"Products with crawled_review_count > 3x declared review_count: {len(large_mismatch)}")
large_mismatch[["id", "product_name", "review_count", "crawled_review_count"]]


Products with crawled_review_count > 3x declared review_count: 53


,id,product_name,review_count,crawled_review_count
6,392842,Bình Đun Siêu Tốc Philips HD9306 (1.5L) - Hàng...,3,170
12,403444,Máy Pha Cà Phê Espresso Tiross TS620 - Hàng Ch...,1,30
18,419160,Máy Xay Thịt Bosch MMR08R2 - Hàng chính hãng,1,28
26,452600,Máy Xay Cầm Tay Panasonic PASO-MX-GS1WRA - Hàn...,1,32
46,548599,Máy Pha Cà Phê Espresso Tiross TS-621 (4 bar) ...,6,35
60,557855,Nồi Áp Suất Đa Năng Sunhouse DNDSHD1552 - 5L (...,2,14
65,558436,Lò Nướng Thủy Tinh Tiger Queen AX-777MV - 11L ...,1,8
74,563737,Nồi Cơm Điện Tử Lock&Lock EJR351BRW (1.8 Lít) ...,3,30
87,792752,Ấm Siêu Tốc Trường Thọ BA 2088 - Xanh (5L)- Hã...,1,5
97,1460199,Bếp Nướng Điện Sunhouse SHD4607 (1500W) - Hàng...,2,30


### 9b. Handle Missing Review Content

Some rows in `reviews` have a `rating` but an empty/missing `content` —
these are rating-only reviews (the customer left a star rating without
writing text), which is a normal, valid pattern on Tiki, not a crawl
error. They still contribute to review-count / rating KPIs, so they are
**not dropped** here.

However, they carry nothing for text-based analysis (BQ6 — Customer
Voice / keyword tagging on comments). A `has_content` flag is created so
downstream notebooks can explicitly exclude these rows from
word-frequency and keyword analysis, without losing them from
review-count-based KPIs.

In [18]:
# Normalize empty-string content to NaN first (crawlers sometimes save
# "" instead of a true null for empty text fields).
reviews["content"] = reviews["content"].replace(r"^\s*$", np.nan, regex=True)

reviews["has_content"] = reviews["content"].notna()

n_no_content = (~reviews["has_content"]).sum()
print(f"Reviews with rating but no written content: {n_no_content} "
      f"({n_no_content / len(reviews) * 100:.1f}%)")

reviews.loc[~reviews["has_content"], ["comment_id", "product_id", "rating"]].head()


Reviews with rating but no written content: 25908 (53.3%)


,comment_id,product_id,rating
73,19047074,359479,5
76,18955149,359479,5
83,18843721,359479,5
85,18785441,359479,5
93,18512424,359479,5


### 9c. Convert Time in Review Dataset

In [19]:
time_cols = ["purchased_at", "created_at"]
for col in time_cols:
    reviews[col] = (
        pd.to_datetime(reviews[col], unit="s", utc=True)
        .dt.tz_convert("Asia/Ho_Chi_Minh")
        .dt.tz_localize(None)
    )

In [20]:
print(reviews[["purchased_at", "created_at"]].dtypes)
print(reviews[["purchased_at", "created_at"]].head(10))

purchased_at    datetime64[ns]
created_at      datetime64[ns]
dtype: object
         purchased_at          created_at
0 2022-01-22 13:58:10 2022-01-23 17:51:06
1 2021-11-11 09:26:06 2021-11-19 15:57:30
2 2021-10-27 15:34:49 2021-10-30 17:07:38
3 2021-08-22 16:55:39 2021-09-04 19:22:34
4 2021-02-09 23:57:42 2021-03-17 15:01:05
5 2020-10-15 14:36:37 2020-10-16 14:42:48
6 2021-09-09 00:31:49 2021-11-28 10:59:09
7 2021-10-10 20:23:57 2021-11-14 02:09:56
8 2021-09-22 19:41:59 2021-10-22 17:11:31
9 2021-09-23 22:03:01 2021-10-11 10:25:02


### 9d. Drop Columns Not Needed for Analysis

Two fields are dropped here, only after having already served their
purpose earlier in this notebook:

- `short_description`: free-text product blurb, not tied to any Business
  Question / KPI in this project's scope.
- `quantity_sold_text`: the raw display string `quantity_sold_value` was
  parsed from — already used above for a sanity check
  (`quantity_sold_text`/`quantity_sold_value` missingness agreement) and
  adds nothing further for analysis.

`all_time_quantity_sold` was already dropped in Section 6, after being
coalesced into `quantity_sold_value` (not discarded outright) to avoid
silently losing the ~4 rows it uniquely covered.

**Important — kept, not dropped:** `quantity_sold_value` (raw, with true
`NaN`) and `quantity_sold_value_filled` (0-filled) look redundant but are
not — the raw column preserves the true "missing vs. zero" distinction
for analyses filtered on `has_sold`, while the filled column is
convenient for aggregations (`groupby(...).sum()`) where `NaN` would
otherwise silently drop rows.

In [21]:
cols_to_drop_final = ["short_description", "quantity_sold_text"]

products = products.drop(columns=[c for c in cols_to_drop_final if c in products.columns])

print("Dropped columns:", cols_to_drop_final)
print("Remaining columns:", products.shape[1])


Dropped columns: ['short_description', 'quantity_sold_text']
Remaining columns: 38


### 9e. Additional Column Audit (based on further manual review)

Candidates spotted during manual review — each is **verified with code
below before being dropped**, not dropped from visual inspection alone
(lesson learned from `all_time_quantity_sold`, which looked identical to
`quantity_sold_value` but actually differed on 4 rows).

- `list_price` vs `original_price`
- `inventory_status`, `inventory_type`
- `stock_item_qty` / `stock_item_min_sale_qty` / `stock_item_max_sale_qty`
- `seller_price` vs `price`
- `seller_is_best_store`

`discount_rate_mismatch` is audited separately below and is **not**
auto-dropped — see rationale in that section.

In [22]:
cols_to_drop_v2 = []

# --- list_price vs original_price ---
# Investigated the 2 mismatched rows (id 234, id 479): in both cases
# price == list_price (not original_price), meaning list_price merely
# duplicates the current selling price and adds no reference-price
# information beyond what original_price already provides. These are
# also the same 2 rows flagged by discount_rate_mismatch below (Tiki's
# own discount_rate field shows 0 despite a real price gap — a genuine
# source-side data issue, not a cleaning bug). Safe to drop list_price.
lp_mismatch = (products["list_price"] != products["original_price"]).sum()
print(f"list_price != original_price: {lp_mismatch} rows (investigated: ids 234, 479 — "
      f"list_price duplicates price in both cases, no unique info lost)")
cols_to_drop_v2.append("list_price")

# --- inventory_status / inventory_type ---
print("\ninventory_status value_counts:")
print(products["inventory_status"].value_counts(dropna=False))
if products["inventory_status"].nunique(dropna=False) <= 1:
    cols_to_drop_v2.append("inventory_status")

print("\ninventory_type value_counts:")
print(products["inventory_type"].value_counts(dropna=False))
if products["inventory_type"].nunique(dropna=False) <= 1:
    cols_to_drop_v2.append("inventory_type")

# --- stock_item_qty / min / max ---
# NOTE: nunique(dropna=False) counts NaN as its own bucket, so a column
# that is constant among non-null rows (e.g. always 1000) but has some
# missing rows would NOT be caught as "constant" by a naive check.
# We check constancy on non-null values separately from missingness,
# and verify whether the missing rows align with `possibly_delisted`
# (already flagged earlier) before deciding it's safe to drop.
for col in ["stock_item_qty", "stock_item_min_sale_qty", "stock_item_max_sale_qty"]:
    print(f"\n{col} value_counts:")
    print(products[col].value_counts(dropna=False).head())

    is_constant_nonnull = products[col].dropna().nunique() <= 1
    missing_matches_delisted = (
        products[col].isna() == products["possibly_delisted"]
    ).all()

    print(f"  Constant among non-null values: {is_constant_nonnull}")
    print(f"  Missing rows exactly match possibly_delisted: {missing_matches_delisted}")

    if is_constant_nonnull and missing_matches_delisted:
        cols_to_drop_v2.append(col)
    elif is_constant_nonnull and not missing_matches_delisted:
        print(f"  -> NOT auto-dropping {col}: constant but missingness doesn't "
              f"align with possibly_delisted, could carry distinct information.")

# --- seller_price vs price ---
# Same NaN pitfall as stock_item_* above: naive (!=) comparison counts
# NaN as a mismatch even when it's just the possibly_delisted rows
# missing seller data entirely. Check the two conditions separately.
sp_real_mismatch = (
    products.loc[products["seller_price"].notna(), "seller_price"]
    != products.loc[products["seller_price"].notna(), "price"]
).sum()
sp_missing_matches_delisted = (
    products["seller_price"].isna() == products["possibly_delisted"]
).all()
print(f"\nseller_price != price (excluding NaN rows): {sp_real_mismatch} rows")
print(f"Missing seller_price rows exactly match possibly_delisted: {sp_missing_matches_delisted}")
if sp_real_mismatch == 0 and sp_missing_matches_delisted:
    cols_to_drop_v2.append("seller_price")  # price already the canonical selling price

# --- seller_is_best_store ---
print("\nseller_is_best_store value_counts:")
print(products["seller_is_best_store"].value_counts(dropna=False))
sbs_constant_nonnull = products["seller_is_best_store"].dropna().nunique() <= 1
sbs_missing_matches_delisted = (
    products["seller_is_best_store"].isna() == products["possibly_delisted"]
).all()
print(f"Constant among non-null values: {sbs_constant_nonnull}")
print(f"Missing rows exactly match possibly_delisted: {sbs_missing_matches_delisted}")
if sbs_constant_nonnull and sbs_missing_matches_delisted:
    cols_to_drop_v2.append("seller_is_best_store")

print("\n" + "=" * 50)
print("Columns confirmed redundant/constant, dropping:", cols_to_drop_v2)
products = products.drop(columns=cols_to_drop_v2)
print("Remaining columns:", products.shape[1])


list_price != original_price: 2 rows (investigated: ids 234, 479 — list_price duplicates price in both cases, no unique info lost)

inventory_status value_counts:
inventory_status
available       1993
out_of_stock       7
upcoming           1
Name: count, dtype: int64

inventory_type value_counts:
inventory_type
backorder       1556
instock          444
discontinued       1
Name: count, dtype: int64

stock_item_qty value_counts:
stock_item_qty
1000.00    1993
NaN           8
Name: count, dtype: int64


  Constant among non-null values: True
  Missing rows exactly match possibly_delisted: True

stock_item_min_sale_qty value_counts:
stock_item_min_sale_qty
1.00    1993
NaN        8
Name: count, dtype: int64
  Constant among non-null values: True
  Missing rows exactly match possibly_delisted: True

stock_item_max_sale_qty value_counts:
stock_item_max_sale_qty
1000.00    1993
NaN           8
Name: count, dtype: int64
  Constant among non-null values: True
  Missing rows exactly match possibly_delisted: True

seller_price != price (excluding NaN rows): 0 rows
Missing seller_price rows exactly match possibly_delisted: True

seller_is_best_store value_counts:
seller_is_best_store
False    1993
NaN         8
Name: count, dtype: int64
Constant among non-null values: True
Missing rows exactly match possibly_delisted: True

Columns confirmed redundant/constant, dropping: ['list_price', 'stock_item_qty', 'stock_item_min_sale_qty', 'stock_item_max_sale_qty', 'seller_price', 'seller_is_best_store

### 9f. Discount Rate Mismatch — Re-check Before Deciding

After tightening the threshold to `1.0pp` in Section 7, if
`discount_rate_mismatch` is still 100% `False`, this could mean one of
two very different things:

1. **A genuine, positive finding** — Tiki's `discount_rate` is always
   internally consistent with `price`/`original_price` within rounding
   tolerance (max diff ~0.5pp). Worth documenting as a clean QA result.
2. **A remaining bug** — if `discount_rate_diff` shows values well above
   1.0pp that still aren't being flagged, something in the comparison
   logic is still off.

The distribution below determines which one it is — **inspect it before
dropping this column.**

In [23]:
print(products["discount_rate_diff"].describe())
print()
print(products["discount_rate_mismatch"].value_counts(dropna=False))

# If describe() confirms max diff is consistent with pure rounding noise
# (roughly <= 0.5-0.6), this is case 1 (genuine clean result) — safe to
# drop the boolean flag itself (zero variance = no analytical value),
# but the finding should still be recorded in cleaning_report.md.
#
# Uncomment the line below only after confirming the distribution above
# actually supports dropping it:

# products = products.drop(columns=["discount_rate_mismatch"])


count   2001.00
mean       0.08
std        0.37
min        0.00
25%        0.00
50%        0.00
75%        0.04
max       15.29
Name: discount_rate_diff, dtype: float64

discount_rate_mismatch
False    1999
True        2
Name: count, dtype: int64


In [24]:
# Investigated the 2 flagged rows:
products.loc[products["discount_rate_mismatch"],
             ["id", "product_name", "price", "original_price",
              "discount_rate", "discount_rate_calc", "discount_rate_diff"]]

# Finding: both rows (id 234 - Nồi cơm điện Tiger JNP-1000, id 479 -
# Bếp Từ Đơn Sunhouse SHD6803) show discount_rate = 0 from the API despite
# a real price gap (up to 15.29pp) between price and original_price.
# This is a genuine SOURCE-side data inconsistency on Tiki's product page
# (their own displayed discount_rate is stale/wrong for these 2 listings),
# not a bug in this cleaning pipeline. discount_rate_calc is more
# trustworthy for these 2 rows. Column kept (not dropped) — it correctly
# flags real issues, not noise.


,id,product_name,price,original_price,discount_rate,discount_rate_calc,discount_rate_diff
234,49601294,Nồi cơm điện Tiger JNP-1000 (Màu trắng) - Hàng...,3290000,3380000,0,2.66,2.66
479,147903536,Bếp Từ Đơn Sunhouse SHD6803 (2000W) - Kèm Nồ...,1008000,1190000,0,15.29,15.29


### 10. Save Cleaned Data

In [25]:
os.makedirs(os.path.dirname(OUT_PRODUCTS), exist_ok=True)

products.to_csv(OUT_PRODUCTS, index=False, encoding="utf-8-sig")
reviews.to_csv(OUT_REVIEWS, index=False, encoding="utf-8-sig")

print(f"Saved cleaned products -> {OUT_PRODUCTS} ({len(products)} rows)")
print(f"Saved cleaned reviews  -> {OUT_REVIEWS} ({len(reviews)} rows)")


Saved cleaned products -> ../data/processed/products_cleaned.csv (2001 rows)
Saved cleaned reviews  -> ../data/processed/reviews_cleaned.csv (48649 rows)


### 11. Cleaning Summary

In [26]:
summary = pd.DataFrame([
    {"metric": "Products (before -> after)",
     "value": f"{n_products_before} -> {len(products)}"},
    {"metric": "Reviews (before -> after)",
     "value": f"{n_reviews_before} -> {len(reviews)}"},
    {"metric": "Columns dropped",
     "value": ", ".join(cols_to_drop)},
    {"metric": "possibly_delisted flagged",
     "value": int(products["possibly_delisted"].sum())},
    {"metric": "discount_rate_mismatch flagged",
     "value": int(products["discount_rate_mismatch"].sum())},
    {"metric": "Large review_count mismatch (>3x)",
     "value": len(large_mismatch)},
    {"metric": "quantity_sold_text/value missingness disagreement",
     "value": int(mismatch_na.sum())},
    {"metric": "Reviews with no written content (rating-only)",
     "value": f"{n_no_content} ({n_no_content / len(reviews) * 100:.1f}%)"},
    {"metric": "Rows rescued from all_time_quantity_sold",
     "value": int(only_all_time_has_value)},
    {"metric": "discount_rate_mismatch threshold used (pp)",
     "value": MISMATCH_THRESHOLD},
    {"metric": "Final columns dropped",
     "value": ", ".join(cols_to_drop_final)},
    {"metric": "subcategory_proxy unclassified rate",
     "value": f"{unclassified_pct:.1f}%"},
    {"metric": "Columns dropped in additional audit (9d)",
     "value": ", ".join(cols_to_drop_v2)},
    {"metric": "Genuine discount_rate anomalies found (9e)",
     "value": f"{int(products['discount_rate_mismatch'].sum())} (ids 234, 479)"},
])

display(summary)


,metric,value
0,Products (before -> after),2001 -> 2001
1,Reviews (before -> after),48649 -> 48649
2,Columns dropped,meta_title
3,possibly_delisted flagged,8
4,discount_rate_mismatch flagged,2
5,Large review_count mismatch (>3x),53
6,quantity_sold_text/value missingness disagreement,4
7,Reviews with no written content (rating-only),25908 (53.3%)
8,Rows rescued from all_time_quantity_sold,4
9,discount_rate_mismatch threshold used (pp),1.00


In [27]:
cleaning_report_lines = [
    "# Data Cleaning Report",
    "",
    "## Summary",
    "",
] + [f"- **{row['metric']}**: {row['value']}" for _, row in summary.iterrows()] + [
    "",
    "## Products Flagged for Manual Review",
    "",
    f"- `possibly_delisted = True` IDs: "
    f"{products.loc[products['possibly_delisted'], 'id'].tolist()}",
    f"- Large review_count mismatch (>3x) IDs: "
    f"{large_mismatch['id'].tolist()}",
]

with open("cleaning_report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaning_report_lines))

print("Saved cleaning_report.md")


Saved cleaning_report.md
